In [2]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"

using System.IO;
using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using Microsoft.Extensions.AI;

var tenantId = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_TENANT_ID");
var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var deploymentName = "gpt-4o";
const string agentName = "expense-agent-AF";
var credOptions = new DefaultAzureCredentialOptions
{
    TenantId = tenantId    
};

// Connect to your project using the endpoint from your project page
AIProjectClient projectClient = new(
    new Uri(endpointUrl),
    new DefaultAzureCredential(credOptions));

// Existing file flow =>
// I have created this index already in Foundry and added the file to it. You can create an index and add files to it using Foundry UI or Azure.AI.Projects SDK.
// In this case, I wanted to use an existing file that I have already uploaded and indexed in Foundry, so I am directly using the vector store content tool with the vector store id to connect it to the agent. The agent can then use this tool to get information from the file when needed.
var fileSearchTool = new HostedFileSearchTool() { Inputs = [new HostedVectorStoreContent("vs_hYPpaDMqc5kbwaHFjSXKWyzI")] };

AIAgent expenseAgent = await projectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, 
instructions: @"You are an AI assistant for corporate expenses.
You answer questions about expenses based on the expenses policy data.
If a user wants to submit an expense claim, you get their email address, a description of the claim, and the amount to be claimed and write the claim details with policy information that you checked to a text file that the user can download. Get the policy information and the email address, description and amount using the tools provided. Do not ask permission to create the file if you have all the needed information.", 
tools: new List<AITool>()
{
    AIFunctionFactory.Create(ExpenseReportTools.GetEmail, "get_email_address"),
    AIFunctionFactory.Create(ExpenseReportTools.GetDescription, "get_description"),
    AIFunctionFactory.Create(ExpenseReportTools.GetAmount, "get_amount"),
    AIFunctionFactory.Create(ExpenseReportTools.AdjustAmount, "adjust_amount"),
    AIFunctionFactory.Create(ExpenseReportTools.CreateFile, "create_file"),
    fileSearchTool
});

AgentSession session = await expenseAgent.CreateSessionAsync();

Console.WriteLine(await expenseAgent.RunAsync("I'd like to submit a claim for a meal.", session));

// Cleanup
await projectClient.Agents.DeleteAgentAsync(expenseAgent.Name);

public static class ExpenseReportTools
{
    public static Task<string> GetEmail() =>
        Task.FromResult("user@xyz.com");

    public static Task<string> GetDescription() =>
        Task.FromResult("This is a description of the expense.");

    public static Task<double> GetAmount() =>
        Task.FromResult(45.00);

    public static Task<string> AdjustAmount() =>
        Task.FromResult("Yes. Adjust the amount to 49.95");

    public static Task<bool> CreateFile(string fileName, string text)
    {        
        File.WriteAllText(fileName, text);
        return Task.FromResult(true);   
    }        
}

Installed Packages Azure.AI.Projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Your meal expense claim has been successfully submitted. You can [download it here](sandbox:/Meal_Expense_Claim.txt).



warning CS1702: Assuming assembly reference 'System.ClientModel, Version=1.8.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.8.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

